In [1]:
import mysql.connector
from tabulate import tabulate

In [26]:
con = mysql.connector.connect(host = "localhost",user="root",password="root",database="ecommerces_online")
res = con.cursor()

In [86]:
customer_ids = []
qry = "select customer_id from customer"
res.execute(qry)
data = res.fetchall()
for i in data:
    customer_ids.append(i[0])
customer_ids

[1, 2, 3, 4, 5]

In [84]:
def get_customerids():
    customer_ids = []
    qry = "select customer_id from customer"
    res.execute(qry)
    data = res.fetchall()
    for i in data:
        customer_ids.append(i[0])
    return customer_ids

In [85]:
get_customerids()

[1, 2, 3, 4, 5]

In [48]:
def get_productids():
    product_ids = []
    qry = "select product_id from product"
    res.execute(qry)
    data = res.fetchall()
    for i in data:
        product_ids.append(i[0])
    return product_ids

In [83]:
get_productids()

[1, 2, 3, 4, 5]

In [56]:
def customer_signup():
    customer_name = input("enter the name:")
    city = input("enter the city:")
    state = input("enter the state")
    qry = '''insert into customer (name,city,state) 
            values (%s,%s,%s)'''
    val = (customer_name,city,state)
    res.execute(qry,val)
    con.commit()
    print("welcome",customer_name)

In [57]:
customer_signup()

enter the name: sri
enter the city: mld
enter the state tn


welcome sri


In [66]:
def stock_entry():
    product_name = input("enter the product_name")
    qunatity = int(input("enter the product quantity"))
    price = int(input("enter the price amount"))
    qry = '''insert into product (product_name,stock_quantity,price)
            values (%s,%s,%s)'''
    val = (product_name,qunatity,price)
    res.execute(qry,val)
    con.commit()
    print("product inserted")

In [67]:
stock_entry()

enter the product_name tv
enter the product quantity 100000
enter the price amount 100000


product inserted


In [68]:
def stock_update(pro,qty):
    qry = "update product set stock_quantity = %s where product_id = %s"
    val = (qty,pro)
    res.execute(qry,val)
    con.commit()
    print("stock updated")

In [70]:
stock_update(3,10)

stock updated


In [72]:
def product_stock(pro):
    qry = "select stock_quantity from product where product_id = %s"
    res.execute(qry,(pro,))
    data = res.fetchall()
    return data[0][0]

In [73]:
product_stock(2)

200

In [74]:
def get_proname(pro_id):
    qry1 = "select product_name from product where product_id = %s"
    val = (pro_id,)
    res.execute(qry1,val)
    productname = res.fetchall()[0][0]
    return productname

In [75]:
 get_proname(4)

'fridge'

In [76]:
def new_orders(pro_id,cust_id,quantity,price):
    proname = get_proname(pro_id)
    total_amount = quantity * price
    qry2 = '''insert into orders 
        (customer_id,product_id,product_name,quantity,price,total_amount)
        values (%s,%s,%s,%s,%s,%s)'''
    val = (cust_id,pro_id,proname,quantity,price,total_amount)
    res.execute(qry2,val)
    con.commit()
    print("order placed")

In [77]:
new_orders(2,1,50,5000)

order placed


In [78]:
def get_price(pro_id):
    qry1 = "select price from product where product_id = %s"
    val = (pro_id,)
    res.execute(qry1,val)
    price = res.fetchall()[0][0]
    return price

In [79]:
get_price(2)

1250000

In [80]:
def get_orders_list():
    orders_list = []
    qry = "select order_id from orders"
    res.execute(qry)
    data = res.fetchall()
    for i in data:
        orders_list.append(i[0])
    return orders_list

In [81]:
get_orders_list()

[1]

In [97]:
def customer_signin():
    customer_id = int(input("enter the customer_id"))
    customer_list = get_customerids()
    if customer_id in customer_list:
        ch = input("1.display\n2.new order\n3.cancel order\n4.customer update details")
        if ch == '1':
            qry = 'select product_id,product_name,price from product'
            res.execute(qry)
            data = res.fetchall()
            print(tabulate(data,headers=["product_id","product_name","price"]))
        elif ch == '2':
            qry = 'select product_id,product_name,price from product'
            res.execute(qry)
            data = res.fetchall()
            print(tabulate(data,headers=['product_id',"product_name","price"]))
            product_id = int(input("kindly enter the product id"))
            product_list = get_productids()
            if product_id in product_list:
                quantity = int(input("enter the quantity"))
                curr_quantity = product_stock(product_id)
                if curr_quantity == 0:
                    print("stock not available")
                elif curr_quantity>=quantity:
                    price = get_price(product_id)
                    new_order(product_id,customer_id,quantity,price)
                    new_qty = curr_quantity - quantity
                    stock_update(product_id,new_qty)
                else:
                    st = product_stock(product_id)
                    print("currently we have",st,"number of qunatity")
                    ch = input("are you go with that press yes else no")
                    if ch == "yes":
                        price = get_price(product_id)
                        new_order(product_id,customer_id,quantity,price)
                        new_qty = curr_quantity - quantity
                        stock_update(product_id,new_qty)
                    else:
                        print("thank you come again")
            else:
                print("kindly check your product id")
        elif ch == '3':
            order_id = int(input("enter the order id"))
            order_list = get_orders_list()
            if order_id in order_list:
                qry1 = "select quantity from orders where order_id = %s"
                val = (order_id,)
                res.execute(qry1,val)
                quantity = res.fetchall()[0][0]
                qry2 = "select product_id from orders where order_id = %s"
                val = (order_id,)
                res.execute(qry2,val)
                product_id = res.fetchall()[0][0]
                curr_stock = product_stock(product_id)
                qry3 = "delete from orders where order_id = %s"
                val = (order_id,)
                res.execute(qry3,val)
                con.commit()
                print("order cancelled")
                new_qty = curr_stock + quantity
                stock_update(product_id,new_qty)
            else:
                print("kindly check your order number")
        elif ch == '4':
        
          name = input("Enter new name: ")
          city = input("Enter city: ")
          state = input("Enter state: ")

          qry = "update customer set name=%s, city=%s, state=%s where customer_id=%s"
          val = (name, city, state, customer_id)

          res.execute(qry, val)
          con.commit()

          print("Updated Successfully")    
            
    else:
        print("kindly check your customer_id")

In [98]:
customer_signin()

enter the customer_id 1
1.display
2.new order
3.cancel order
4.customer update details 4
Enter new name:  pooja
Enter city:  cmb
Enter state:  tn


Updated Successfully
